In [ ]:
%%writefile /kaggle/working/mono_train.py

import os

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from kaggle_secrets import UserSecretsClient
from huggingface_hub import notebook_login, whoami

SECRET_LABEL= "HF_TOKEN"
MODEL_NAME = "google/gemma-3-270m"
DATASET_REPO = "davron04/dueta-tokenized"
OUTPUT_DIR = "/kaggle/working/gemma-3-270m-dueta-base-adapter"
REMOTE_MODEL = "davron04/gemma-3-270m-dueta-base-adapter"
SEED = 42
EPOCHS = 1
NUM_DEVICES = 2
TRAIN_BATCH_SIZE = 8
PER_DEVICE_TRAIN_BATCH_SIZE = 4
PER_DEVICE_EVAL_BATCH_SIZE = 4
EVAL_INTERVAL_RATIO = 0.09
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
LORA_RANK = 16
LORA_ALPHA = LORA_RANK * 2
LORA_TARGET_MODULES = "all-linear"
LORA_DROPOUT = 0.05
LORA_BIAS = "none"
LORA_TASK_TYPE = "CAUSAL_LM"
LOG_STEPS = 500

In [ ]:
%%writefile -a /kaggle/working/mono_train.py

import torch
import numpy as np

class CustomDataCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id
    def __call__(self, samples):
        num_of_samples = len(samples)
        lengths = [sample["length"] for sample in samples]
        max_length = max(lengths) - 1

        input_ids_batch = np.zeros((num_of_samples, max_length), dtype=np.int64)
        attention_mask_batch = np.zeros((num_of_samples, max_length), dtype=np.int64)
        labels_batch = np.zeros((num_of_samples, max_length), dtype=np.int64)

        for i, sample in enumerate(samples):
            seq_length = sample["length"]

            input_ids = [self.pad_token_id] * max_length
            input_ids[:seq_length - 1] = sample["input_ids"][:-1]
            input_ids_batch[i] = input_ids

            attention_mask = [0] * max_length
            attention_mask[:seq_length - 1] = [1] * (seq_length - 1)
            attention_mask_batch[i] = attention_mask

            labels = [-100] * max_length
            labels[:seq_length - 1] = sample["input_ids"][1:]
            labels_batch[i] = labels

        return {
            "input_ids": torch.tensor(input_ids_batch, dtype=torch.int64),
            "attention_mask": torch.tensor(attention_mask_batch, dtype=torch.int64),
            "labels": torch.tensor(labels_batch, dtype=torch.int64),
        }

In [ ]:
%%writefile -a /kaggle/working/mono_train.py

import torch
from torch.nn.functional import cross_entropy
from transformers import Trainer

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(
            input_ids = inputs['input_ids'],
            attention_mask = inputs['attention_mask']
        )
        logits=outputs.logits
        labels = inputs.get("labels")
        loss = cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
%%writefile -a /kaggle/working/mono_train.py

import torch
from torch.nn.functional import cross_entropy

class MetricsTracker:
    def __init__(self):
        self.losses = []

    def __call__(self, eval_preds, compute_result=False):
        logits, labels = eval_preds

        loss = cross_entropy(
            logits.view(-1, logits.size(-1)),
            labels.view(-1)
        )
        self.losses.append(loss.unsqueeze(0))

        if not compute_result:
            return {}

        avg_loss = torch.mean(torch.cat(self.losses))
        try:
            perplexity = torch.exp(avg_loss)
        except OverflowError:
            perplexity = float("inf")

        self.losses.clear()
        return {
            "eval_loss": avg_loss.item(),
            "eval_perplexity": perplexity.item() if perplexity != float("inf") else perplexity,
        }

In [ ]:
%%writefile -a /kaggle/working/mono_train.py

from accelerate import PartialState
from transformers import TrainingArguments
from huggingface_hub import snapshot_download, HfApi
from peft import LoraConfig, get_peft_model, PeftModel

def get_remote_checkpoint():
    api = HfApi()
    try:
        repo_tree = api.list_repo_tree(
            repo_id = REMOTE_MODEL,
            path_in_repo = "last-checkpoint",
            repo_type = "model"
        )
        list(repo_tree)
        snapshot_download(
            repo_id = REMOTE_MODEL,
            repo_type = "model",
            local_dir = OUTPUT_DIR
        )
        return True
    except:
        return False


def train_function():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    hf_token = UserSecretsClient().get_secret(SECRET_LABEL)
    os.environ["HF_TOKEN"] = hf_token
    notebook_login()

    train_from_checkpoint = False
    train_from_checkpoint = get_remote_checkpoint()

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    train_set = load_dataset(DATASET_REPO, split="train", streaming=False)
    train_set = train_set.filter(lambda x: x["length"] <= 257)
    val_set = load_dataset(DATASET_REPO, split="test", streaming=False)
    val_set = val_set.filter(lambda x: x["length"] <= 257)
    
    data_collator = CustomDataCollator(pad_token_id=tokenizer.pad_token_id)
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.float32
    )
    if train_from_checkpoint:
        lora_model = PeftModel.from_pretrained(base_model, REMOTE_MODEL, is_trainable=True)
    else:
        lora_config = LoraConfig(
            r=LORA_RANK,
            lora_alpha=LORA_ALPHA,
            target_modules=LORA_TARGET_MODULES,
            lora_dropout=LORA_DROPOUT,
            bias=LORA_BIAS,
            task_type=LORA_TASK_TYPE,
        )
        lora_model = get_peft_model(base_model, lora_config)
    lora_model.print_trainable_parameters()
    
    evaluate = MetricsTracker()
    
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,

        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        num_train_epochs=EPOCHS,

        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_steps=WARMUP_RATIO,

        optim="adamw_torch",
        weight_decay=WEIGHT_DECAY,

        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        average_tokens_across_devices=False,
        max_grad_norm=1.0,

        fp16=True,
        fp16_full_eval=True,

        torch_compile=False,

        logging_strategy="steps",
        logging_steps=LOG_STEPS,
        logging_first_step=True,
        logging_nan_inf_filter=False,
        log_level="info",
        disable_tqdm=False,

        eval_strategy="steps",
        eval_steps = EVAL_INTERVAL_RATIO,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        eval_on_start=False,
        batch_eval_metrics=True,

        save_strategy="steps",
        save_steps=EVAL_INTERVAL_RATIO,
        save_total_limit=15,

        push_to_hub=True,
        hub_token=hf_token,
        hub_private_repo=False,
        hub_model_id=REMOTE_MODEL,
        hub_strategy="checkpoint",

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        seed=SEED,
        data_seed=SEED,

        dataloader_drop_last=True,
        remove_unused_columns = False,
        label_names=["labels"],
        train_sampling_strategy = "random",
        length_column_name = "length",

        ddp_backend="nccl",
        ddp_find_unused_parameters=False,
    )
            
    trainer = CustomTrainer(
        model=lora_model,
        args=training_args,
        data_collator=data_collator,
        train_dataset=train_set,
        eval_dataset=val_set,
        compute_metrics=evaluate,
    )
    if train_from_checkpoint:
        trainer.train(resume_from_checkpoint=f"{OUTPUT_DIR}/last-checkpoint")
    else:
        trainer.train()
    trainer.push_to_hub()

In [ ]:
%%writefile -a /kaggle/working/mono_train.py

from transformers import Trainer

import torch.distributed as dist

if __name__ == "__main__":
    try:
        train_function()
    finally:
        if dist.is_available() and dist.is_initialized():
            dist.destroy_process_group()

In [ ]:
!torchrun --nproc_per_node=2 /kaggle/working/mono_train.py